# Cross-Resolution Robustness Check

This notebook re-evaluates the **hourly-selected** series under the **10-minute** criteria
(as a cross-resolution robustness check). Because finer aggregation depresses STL strength,
breaks long-range autocorrelation, and lowers activity ratios, hourly-selected series do not
all pass at 10-min — which is exactly why the methodology runs **independent selection
pipelines per resolution** (`selected_ids/hourly/` and `selected_ids/10min/`). The 10-min
benchmarks shipped in `selected_ids/10min/` are derived independently and are not validated
here.

In [1]:
import pandas as pd

from config import AGGREGATIONS, BENCHMARKS
from utils import load_dataset
from scoring import SCORE_FUNCTIONS

AGGREGATION = "10min"
AGG_PARAMS = AGGREGATIONS[AGGREGATION]
BENCHMARK_NAMES = ["SEASON", "DRIFT", "PERIODIC_SPACES", "WORKERS", "RANDOM"]

In [2]:
# Load hourly selected IDs and group by source type.
# IPs from SEASON and WORKERS came from the full IP dataset; the rest from the 1000-IP sample.
selected = {b: pd.read_csv(f"selected_ids/hourly/{b}.csv") for b in BENCHMARK_NAMES}
for b, df in selected.items():
    print(f"{b}: {len(df)} series")

LEVEL_TO_SOURCE = {"Institutions": "institutions", "Subnets": "subnets", "IPs": "ips"}
IPS_FULL_BENCHMARKS = {"SEASON", "WORKERS"}

ids_per_source = {"institutions": set(), "subnets": set(), "ips": set(), "ips_full": set()}
for b, df in selected.items():
    for _, row in df.iterrows():
        src = LEVEL_TO_SOURCE[row["level"]]
        if src == "ips" and b in IPS_FULL_BENCHMARKS:
            ids_per_source["ips_full"].add(int(row["ts_id"]))
        else:
            ids_per_source[src].add(int(row["ts_id"]))

for src, ids in ids_per_source.items():
    if ids:
        print(f"{src}: {len(ids)} unique IDs to load")

SEASON: 75 series
DRIFT: 75 series
PERIODIC_SPACES: 25 series
WORKERS: 25 series
RANDOM: 63 series
institutions: 58 unique IDs to load
subnets: 72 unique IDs to load
ips: 73 unique IDs to load
ips_full: 50 unique IDs to load


In [3]:
data = {}
for src, ids in ids_per_source.items():
    if ids:
        data[src] = load_dataset(
            src,
            aggregation=AGG_PARAMS["enum"],
            time_range=AGG_PARAMS["time_range"],
            ts_ids=list(ids),
        )

100%|██████████████████████████| 58/58 [00:07<00:00,  7.67it/s]



Config Details
    Used for database: CESNET-TimeSeries24
    Aggregation: AgreggationType.AGG_10_MINUTES
    Source: SourceType.INSTITUTIONS

    Time series
        Time series IDS: [256 259 132 261   4 ... 118 249 123 252 127], Length=58
    Time periods
        Train time periods: range(0, 40298)
        Val time periods: None
        Test time periods: None
        All time periods: range(0, 40298)
    Features
        Taken features: ['n_bytes']
        Default values: [0.]
        Time series ID included: True
        Time included: True    
        Time format: TimeFormat.DATETIME
    Sliding window
        Sliding window size: None
        Sliding window prediction size: None
        Sliding window step size: 1
        Set shared size: 0
    Fillers
        Filler type: NoFiller
    Transformers
        Transformer type: NoTransformer
    Anomaly handler
        Anomaly handler type: NoAnomalyHandler        
    Batch sizes
        Train batch size: 32
        Val batch size:

100%|██████████████████████████| 72/72 [00:07<00:00,  9.43it/s]



Config Details
    Used for database: CESNET-TimeSeries24
    Aggregation: AgreggationType.AGG_10_MINUTES
    Source: SourceType.INSTITUTION_SUBNETS

    Time series
        Time series IDS: [509 388 133 261 391 ... 498 373 501 532 381], Length=72
    Time periods
        Train time periods: range(0, 40298)
        Val time periods: None
        Test time periods: None
        All time periods: range(0, 40298)
    Features
        Taken features: ['n_bytes']
        Default values: [0.]
        Time series ID included: True
        Time included: True    
        Time format: TimeFormat.DATETIME
    Sliding window
        Sliding window size: None
        Sliding window prediction size: None
        Sliding window step size: 1
        Set shared size: 0
    Fillers
        Filler type: NoFiller
    Transformers
        Transformer type: NoTransformer
    Anomaly handler
        Anomaly handler type: NoAnomalyHandler        
    Batch sizes
        Train batch size: 32
        Val batc

100%|██████████████████████████| 73/73 [00:07<00:00,  9.37it/s]



Config Details
    Used for database: CESNET-TimeSeries24
    Aggregation: AgreggationType.AGG_10_MINUTES
    Source: SourceType.IP_ADDRESSES_SAMPLE

    Time series
        Time series IDS: [ 149504 1512833  300547  355332  650373 ... 170487 156921 792059 121596 237694], Length=73
    Time periods
        Train time periods: range(0, 40298)
        Val time periods: None
        Test time periods: None
        All time periods: range(0, 40298)
    Features
        Taken features: ['n_bytes']
        Default values: [0.]
        Time series ID included: True
        Time included: True    
        Time format: TimeFormat.DATETIME
    Sliding window
        Sliding window size: None
        Sliding window prediction size: None
        Sliding window step size: 1
        Set shared size: 0
    Fillers
        Filler type: NoFiller
    Transformers
        Transformer type: NoTransformer
    Anomaly handler
        Anomaly handler type: NoAnomalyHandler        
    Batch sizes
        Tr

100%|██████████████████████████| 50/50 [00:07<00:00,  6.65it/s]



Config Details
    Used for database: CESNET-TimeSeries24
    Aggregation: AgreggationType.AGG_10_MINUTES
    Source: SourceType.IP_ADDRESSES_FULL

    Time series
        Time series IDS: [258691 472452 514692   3846 139017 ...   3313   6005  61048 119674 114939], Length=50
    Time periods
        Train time periods: range(0, 40298)
        Val time periods: None
        Test time periods: None
        All time periods: range(0, 40298)
    Features
        Taken features: ['n_bytes']
        Default values: [0.]
        Time series ID included: True
        Time included: True    
        Time format: TimeFormat.DATETIME
    Sliding window
        Sliding window size: None
        Sliding window prediction size: None
        Sliding window step size: 1
        Set shared size: 0
    Fillers
        Filler type: NoFiller
    Transformers
        Transformer type: NoTransformer
    Anomaly handler
        Anomaly handler type: NoAnomalyHandler        
    Batch sizes
        Train bat

In [4]:
def _resolve_source(level, ts_id):
    src = LEVEL_TO_SOURCE[level]
    if src == "ips" and ts_id in ids_per_source.get("ips_full", set()):
        src = "ips_full"
    return src


def _score_one(level, ts_id, benchmark, **extra):
    d = data[_resolve_source(level, ts_id)]
    grp = d["df"][d["df"][d["id_col"]] == ts_id]
    return SCORE_FUNCTIONS[benchmark](
        grp, d["id_col"], AGG_PARAMS, BENCHMARKS[benchmark], **extra
    )


def _pool_pass(scores_df, benchmark):
    bp = BENCHMARKS[benchmark]
    smin, smax = bp["sparsity_min"], bp.get("sparsity_max")
    sp = (
        scores_df["ratio_active"].between(smin, smax)
        if smax is not None
        else scores_df["ratio_active"] >= smin
    )
    op, col, thr = bp["metric_op"], bp["metric_col"], bp["threshold"]
    if op == ">=":
        m = scores_df[col] >= thr
    elif op == "<=":
        m = scores_df[col] <= thr
    elif op == "custom":
        m = scores_df["has_drift"] & scores_df["drift_in_test"]
    else:
        raise ValueError(f"Unknown metric_op: {op}")
    return sp & m


def validate(benchmark, **extra):
    """Re-evaluate the hourly-selected series for `benchmark` under the 10-min criteria."""
    rows = []
    for _, r in selected[benchmark].iterrows():
        ts_id = int(r["ts_id"])
        s = _score_one(r["level"], ts_id, benchmark, **extra)
        s = {k: v for k, v in s.items() if not k.startswith("id_")}
        rows.append({"level": r["level"], "ts_id": ts_id, **s})
    df = pd.DataFrame(rows)
    df["pass"] = _pool_pass(df, benchmark)
    return df


def _print_summary(df, benchmark):
    n_pass = int(df["pass"].sum())
    print(f"{benchmark}: {n_pass}/{len(df)} pass at 10-min")

## SEASON — seasonality strength >= 0.7

In [5]:
df_season = validate("SEASON")
_print_summary(df_season, "SEASON")
df_season[~df_season["pass"]]

SEASON: 28/75 pass at 10-min


,level,ts_id,ratio_active,strength_daily,strength_weekly,max_strength,pass
1,Institutions,16,0.999603,0.589686,0.682485,0.682485,False
2,Institutions,38,0.989652,0.625642,0.646279,0.646279,False
7,Institutions,70,0.985111,0.563890,0.612073,0.612073,False
8,Institutions,96,1.000000,0.650725,0.670135,0.670135,False
9,Institutions,112,0.999131,0.608734,0.661237,0.661237,False
10,Institutions,123,0.999926,0.653318,0.689183,0.689183,False
11,Institutions,132,0.991712,0.545112,0.590180,0.590180,False
13,Institutions,158,0.996104,0.608727,0.653241,0.653241,False
14,Institutions,183,0.996054,0.596404,0.695297,0.695297,False
16,Institutions,195,0.999975,0.575192,0.630696,0.630696,False


## DRIFT — sustained MA deviation > 0.30 for >= 14 days, visible in test

In [6]:
df_drift = validate("DRIFT")
_print_summary(df_drift, "DRIFT")
df_drift[~df_drift["pass"]]

DRIFT: 57/75 pass at 10-min


,level,ts_id,ratio_active,has_drift,drift_in_test,max_rel_dev,drift_fraction,n_drift_events,pass
16,Institutions,172,0.870043,True,True,0.938039,0.006948,1,False
17,Institutions,182,0.998139,True,False,1.387059,0.008412,2,False
24,Institutions,269,0.875527,True,True,0.810757,0.003226,1,False
28,Subnets,96,0.412105,True,True,0.812912,0.090823,1,False
29,Subnets,133,0.684997,False,False,0.965946,0.000000,0,False
42,Subnets,457,0.499553,True,True,1.097739,0.034295,2,False
45,Subnets,477,0.163854,True,True,1.261581,0.036329,2,False
48,Subnets,523,0.000000,False,False,0.000000,0.000000,0,False
49,Subnets,533,0.735595,True,True,1.547425,0.083379,4,False
58,IPs,43611,0.858876,True,True,2.281285,0.117351,3,False


## PERIODIC_SPACES — ACF peak >= 0.6 on binary indicator

In [7]:
df_ps = validate("PERIODIC_SPACES")
_print_summary(df_ps, "PERIODIC_SPACES")
df_ps[~df_ps["pass"]]

PERIODIC_SPACES: 9/25 pass at 10-min


,level,ts_id,ratio_active,max_acf,dominant_lag,pass
0,IPs,101,0.159958,0.580189,1008,False
5,IPs,11212,0.370242,0.492822,1008,False
6,IPs,35748,0.203087,0.551024,144,False
7,IPs,121748,0.102486,0.384401,1008,False
8,IPs,162890,0.055710,0.569637,1008,False
9,IPs,165515,0.044667,0.405847,1008,False
10,IPs,232476,0.223113,0.573315,1008,False
11,IPs,237694,0.060474,0.595550,1008,False
12,IPs,302806,0.143158,0.479919,1008,False
13,IPs,307400,0.497047,0.561344,144,False


## WORKERS — WHR >= 0.6

In [8]:
# WORKERS needs the weekends/holidays calendar — pull it from any loaded dataset.
wh = next(iter(data.values()))["dataset"].get_additional_data("weekends_and_holidays").copy()
wh["day"] = pd.to_datetime(wh["Date"]).dt.normalize()
non_work_days = set(wh["day"])
print(f"Non-working days in calendar: {len(non_work_days)}")

df_workers = validate("WORKERS", non_work_days=non_work_days)
_print_summary(df_workers, "WORKERS")
df_workers[~df_workers["pass"]]

Non-working days in calendar: 89
WORKERS: 7/25 pass at 10-min


,level,ts_id,ratio_active,whr,total_active_intervals,work_active_intervals,pass
1,IPs,21943,0.187181,0.740554,7543,5586,False
2,IPs,61466,0.184103,0.618817,7419,4591,False
3,IPs,71574,0.155864,0.615507,6281,3866,False
4,IPs,142157,0.213087,0.608478,8587,5225,False
6,IPs,222483,0.205420,0.703189,8278,5821,False
8,IPs,286014,0.200754,0.610507,8090,4939,False
9,IPs,288481,0.200655,0.615632,8086,4978,False
11,IPs,308900,0.211350,0.605025,8517,5153,False
12,IPs,328623,0.184401,0.620374,7431,4610,False
13,IPs,365864,0.193459,0.635839,7796,4957,False


## RANDOM — max |ACF| <= 0.1

In [9]:
df_random = validate("RANDOM")
_print_summary(df_random, "RANDOM")
df_random[~df_random["pass"]]

RANDOM: 34/63 pass at 10-min


,level,ts_id,ratio_active,max_acf,pass
3,Institutions,256,0.262693,0.667786,False
5,Institutions,264,0.122810,0.142893,False
6,Institutions,267,0.000000,inf,False
9,Institutions,277,0.364807,0.062230,False
10,Institutions,279,0.000000,inf,False
11,Institutions,280,0.122016,0.010075,False
13,Subnets,23,0.000000,inf,False
16,Subnets,45,0.999653,0.400802,False
17,Subnets,46,0.272942,0.143775,False
19,Subnets,57,0.000000,inf,False


## Summary

In [10]:
summary = pd.DataFrame([
    {"Benchmark": b, "Total": len(df), "Pass": int(df["pass"].sum())}
    for b, df in [
        ("SEASON", df_season),
        ("DRIFT", df_drift),
        ("PERIODIC_SPACES", df_ps),
        ("WORKERS", df_workers),
        ("RANDOM", df_random),
    ]
])
summary["Fail"] = summary["Total"] - summary["Pass"]
summary["Pass %"] = (summary["Pass"] / summary["Total"] * 100).round(1)
print("Hourly-selection survival under 10-minute criteria")
print("=" * 50)
summary

Hourly-selection survival under 10-minute criteria


,Benchmark,Total,Pass,Fail,Pass %
0,SEASON,75,28,47,37.3
1,DRIFT,75,57,18,76.0
2,PERIODIC_SPACES,25,9,16,36.0
3,WORKERS,25,7,18,28.0
4,RANDOM,63,34,29,54.0
